In [11]:
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import SystemMessagePromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel
from dotenv import load_dotenv
load_dotenv()

True

In [12]:
class SQL_Query(BaseModel):
    query: str
    is_malicious: bool
    table_count: int


In [13]:
# ponytail: no temperature — Opus 4.8 rejects it (400). Key read from ANTHROPIC_API_KEY.
model = ChatAnthropic(model="claude-opus-4-8", max_tokens=1024)

In [14]:
template = """
You are a cybersecurity assistant that generates SQL queries to test whether a specific system is vulnerable to SQL injection attacks.

You will be provided with the a database table named {table} and a list of column names:
{columns}

Only use this information to generate a SQL query to test whether the application using this database is vulnerable to SQL injection attacks.
The query should range from very simple to very complex.  Include examples with joins, code execution and unions.

{format_instructions}
"""

In [15]:
json_output_parser = PydanticOutputParser(pydantic_object=SQL_Query)

In [16]:
print(json_output_parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"query": {"title": "Query", "type": "string"}, "is_malicious": {"title": "Is Malicious", "type": "boolean"}, "table_count": {"title": "Table Count", "type": "integer"}}, "required": ["query", "is_malicious", "table_count"]}
```


In [17]:
# Define the messages. Anthropic requires at least one human turn (system-only is rejected),
# so pair the system instruction with a short human message.
system_message = SystemMessagePromptTemplate.from_template(template)
chat_prompt = ChatPromptTemplate.from_messages([
    system_message,
    ("human", "Generate the SQL query now."),
])

In [18]:
# Build the chain
chain = chat_prompt | model | json_output_parser
result = chain.invoke({"table": "users",
                       "columns": ["id", "name", "email", "password", "phone_number"],
                       "format_instructions": json_output_parser.get_format_instructions()
                       },
                      )

In [19]:
print(result)

query="SELECT id, name, email FROM users WHERE id = 1 OR 1=1; -- ' UNION SELECT id, name, password FROM users WHERE '1'='1'" is_malicious=True table_count=1
